# Inicialização e Validação do Ambiente Analítico

Preparação e estruturação do ambiente Databricks para o processamento do case da V-Credit. Esta etapa importa as configurações globais, assegura a criação idempotente da arquitetura de armazenamento (catálogo, schemas das camadas medalhão e volume da Landing Zone) e executa a verificação prévia de integridade dos arquivos de entrada obrigatórios.

In [0]:
%run ./00_Config

## Criação da Infraestrutura

Criação programática do catálogo, schemas e volume de dados utilizando diretrizes de idempotência (`IF NOT EXISTS`). Essa abordagem previne conflitos de concorrência em execuções colaborativas e assegura a reprodutibilidade do ambiente por qualquer membro da equipe.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.{landing_volume_name}")

print(f"Catálogo: {catalog}")
print(f"Schemas: {bronze_schema}, {silver_schema}, {gold_schema}")
print(f"Landing Zone: {landing_path}")

## Verificação de Dependências na Landing Zone

Auditoria pré-execução para validar a presença dos arquivos de origem disponibilizados pela V-Credit. O processo interrompe a execução caso algum insumo obrigatório esteja ausente, mitigando falhas decorrentes de lacunas no upload de dados.

In [0]:
expected_files = {
    "base_atendentes.csv", "base_motivos.csv", "canais.csv",
    "chamados.csv", "Chamados_Hora.CSV", "clientes.csv",
    "custos.csv", "pesquisa_satisfacao.csv",
}

existing_files = {file.name for file in dbutils.fs.ls(landing_path)}
missing_files = sorted(expected_files - existing_files)
unexpected_files = sorted(existing_files - expected_files)

assert not missing_files, (
    "Landing Zone incompleta. Arquivos obrigatórios ausentes: " + ", ".join(missing_files)
)

print(f"[OK] {len(expected_files)} arquivos obrigatórios encontrados com sucesso.")
if unexpected_files:
    print(f"[AVISO] Arquivos adicionais identificados serão ignorados: {unexpected_files}")